In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb


DATA_DIR = Path("data/yearly")
ARTIFACT_DIR = Path("artifacts/probability_calibration")

TRAIN_YEARS = (2020, 2021, 2022, 2023)
VALIDATION_YEAR = 2024

TARGET = "crime_count"
DATETIME_COLUMN = "cmplnt_fr_dt"

CATEGORICAL_FEATURES = [
    "grid_id",
    "time_period"
]

SAMPLE_INDEX_PATH = Path(
    "artifacts/model_family_comparison/sampled_indices_2020_2023.npz"
)

BATCH_SIZE = 500_000

In [4]:
sample_indices = np.load(SAMPLE_INDEX_PATH)

train_parts = []

for year in TRAIN_YEARS:
    df = pd.read_parquet(yearly_path(year))
    train_parts.append(df.iloc[sample_indices[f"year_{year}"]])

train_sample = pd.concat(
    train_parts,
    ignore_index=True
)

train_sample.shape

(1000000, 16)

In [5]:
X_train = train_sample.drop(
    columns=[TARGET, DATETIME_COLUMN]
)

y_train = train_sample[TARGET]

print(X_train.shape)
print(y_train.shape)

(1000000, 14)
(1000000,)


In [6]:
model = xgb.XGBRegressor(
    objective="count:poisson",
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train
)

,"objective objective: typing.Union[str, xgboost.objective.Objective, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'count:poisson'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.9
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets impor

In [7]:
print(model)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=-1, num_parallel_tree=None, ...)


In [8]:
val_batch = pd.read_parquet(
    yearly_path(VALIDATION_YEAR)
).iloc[:BATCH_SIZE]

X_val_batch = val_batch.drop(
    columns=[TARGET, DATETIME_COLUMN]
)

predicted_count = model.predict(X_val_batch)

print(predicted_count[:10])
print(predicted_count.min())
print(predicted_count.max())
print(predicted_count.mean())

[1.0499606e-04 7.2616647e-05 6.9567002e-05 6.9308342e-05 7.1454706e-05
 7.3100178e-05 7.7438512e-05 8.4174149e-05 8.6276028e-05 8.8820532e-05]
3.727391e-05
0.19147405
0.008626023


In [9]:
bin_edges = np.arange(0.0, 1.1, 0.1)

calibration = [
    {
        "rows": 0,
        "predicted_probability_sum": 0.0,
        "actual_positive": 0
    }
    for _ in range(len(bin_edges) - 1)
]

validation_path = yearly_path(VALIDATION_YEAR)

for start in range(0, 8_309_664, BATCH_SIZE):
    end = min(start + BATCH_SIZE, 8_309_664)

    batch_df = pd.read_parquet(
        validation_path
    ).iloc[start:end]

    X_batch = batch_df.drop(
        columns=[TARGET, DATETIME_COLUMN]
    )

    predicted_count = model.predict(X_batch)

    predicted_probability = (
        1 - np.exp(-predicted_count)
    )

    actual_positive = (
        batch_df[TARGET].to_numpy() >= 1
    )

    bin_ids = np.digitize(
        predicted_probability,
        bin_edges,
        right=False
    ) - 1

    bin_ids = np.clip(
        bin_ids,
        0,
        len(calibration) - 1
    )

    for bin_id in range(len(calibration)):
        mask = bin_ids == bin_id

        if not np.any(mask):
            continue

        calibration[bin_id]["rows"] += int(mask.sum())

        calibration[bin_id]["predicted_probability_sum"] += (
            predicted_probability[mask].sum()
        )

        calibration[bin_id]["actual_positive"] += int(
            actual_positive[mask].sum()
        )

    print(
        f"Processed rows {start:,} to {end:,}"
    )

Processed rows 0 to 500,000
Processed rows 500,000 to 1,000,000
Processed rows 1,000,000 to 1,500,000
Processed rows 1,500,000 to 2,000,000
Processed rows 2,000,000 to 2,500,000
Processed rows 2,500,000 to 3,000,000
Processed rows 3,000,000 to 3,500,000
Processed rows 3,500,000 to 4,000,000
Processed rows 4,000,000 to 4,500,000
Processed rows 4,500,000 to 5,000,000
Processed rows 5,000,000 to 5,500,000
Processed rows 5,500,000 to 6,000,000
Processed rows 6,000,000 to 6,500,000
Processed rows 6,500,000 to 7,000,000
Processed rows 7,000,000 to 7,500,000
Processed rows 7,500,000 to 8,000,000
Processed rows 8,000,000 to 8,309,664


In [10]:
calibration_rows = []

for i in range(len(calibration)):
    rows = calibration[i]["rows"]

    if rows == 0:
        continue

    lower = bin_edges[i]
    upper = bin_edges[i + 1]

    mean_predicted_probability = (
        calibration[i]["predicted_probability_sum"] / rows
    )

    actual_rate = (
        calibration[i]["actual_positive"] / rows
    )

    calibration_rows.append({
        "Probability Range": f"{lower:.0%}–{upper:.0%}",
        "Rows": rows,
        "Mean Predicted Probability": mean_predicted_probability,
        "Actual Crime Rate": actual_rate
    })

calibration_df = pd.DataFrame(calibration_rows)

calibration_df

,Probability Range,Rows,Mean Predicted Probability,Actual Crime Rate
0,0%–10%,6420474,0.026332,0.023212
1,10%–20%,1149298,0.142230,0.120378
2,20%–30%,475940,0.242954,0.216752
3,30%–40%,186443,0.341595,0.326674
4,40%–50%,62420,0.439539,0.432233
5,50%–60%,11594,0.531728,0.536139
6,60%–70%,2800,0.656022,0.750714
7,70%–80%,695,0.723882,0.804317


In [11]:
train_files = [
    yearly_path(year)
    for year in TRAIN_YEARS
]

for path in train_files:
    df_info = pd.read_parquet(path)
    print(
        path.name,
        df_info.shape,
        f"{df_info.memory_usage(deep=True).sum() / 1024**3:.2f} GB"
    )
    del df_info

citysense_2020.parquet (8309664, 16) 0.43 GB
citysense_2021.parquet (8286960, 16) 0.42 GB
citysense_2022.parquet (8286960, 16) 0.42 GB
citysense_2023.parquet (8286960, 16) 0.42 GB


In [12]:
train_parts = []

for year in TRAIN_YEARS:
    df = pd.read_parquet(
        yearly_path(year)
    )

    train_parts.append(df)

    print(
        f"{year}: {df.shape}"
    )

train_full = pd.concat(
    train_parts,
    ignore_index=True
)

del train_parts

print("Full training data:", train_full.shape)

2020: (8309664, 16)
2021: (8286960, 16)
2022: (8286960, 16)
2023: (8286960, 16)
Full training data: (33170544, 16)


In [13]:
X_train = train_full.drop(
    columns=[TARGET, DATETIME_COLUMN]
)

y_train = train_full[TARGET]

print(X_train.shape)
print(y_train.shape)

(33170544, 14)
(33170544,)


In [14]:
for year in TRAIN_YEARS:
    df = pd.read_parquet(
        yearly_path(year),
        columns=[TARGET]
    )
    print(year, len(df))
    del df

2020 8309664
2021 8286960
2022 8286960
2023 8286960


In [16]:
model = xgb.XGBRegressor(
    objective="count:poisson",
    n_estimators=600,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=20,
    subsample=0.9,
    colsample_bytree=0.9,
    tree_method="hist",
    enable_categorical=True,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train
)

print("Full-data model trained successfully.")

Full-data model trained successfully.


In [19]:
import pyarrow.dataset as ds

validation_path = yearly_path(VALIDATION_YEAR)
dataset = ds.dataset(validation_path, format="parquet")

In [20]:
processed_rows = 0

for batch in dataset.to_batches(batch_size=BATCH_SIZE):
    batch_df = batch.to_pandas()

    X_batch = batch_df.drop(
        columns=[TARGET, DATETIME_COLUMN]
    )

    predicted_count = model.predict(X_batch)

    predicted_probability = (
        1 - np.exp(-predicted_count)
    )

    actual_positive = (
        batch_df[TARGET].to_numpy() >= 1
    )

    bin_ids = np.digitize(
        predicted_probability,
        bin_edges,
        right=False
    ) - 1

    bin_ids = np.clip(
        bin_ids,
        0,
        len(calibration) - 1
    )

    for bin_id in range(len(calibration)):
        mask = bin_ids == bin_id

        if not np.any(mask):
            continue

        calibration[bin_id]["rows"] += int(mask.sum())
        calibration[bin_id]["predicted_probability_sum"] += (
            predicted_probability[mask].sum()
        )
        calibration[bin_id]["actual_positive"] += int(
            actual_positive[mask].sum()
        )

    processed_rows += len(batch_df)

    print(f"Processed {processed_rows:,} / 8,309,664")

Processed 500,000 / 8,309,664
Processed 878,400 / 8,309,664
Processed 1,378,400 / 8,309,664
Processed 1,756,800 / 8,309,664
Processed 2,256,800 / 8,309,664
Processed 2,635,200 / 8,309,664
Processed 3,135,200 / 8,309,664
Processed 3,513,600 / 8,309,664
Processed 4,013,600 / 8,309,664
Processed 4,392,000 / 8,309,664
Processed 4,892,000 / 8,309,664
Processed 5,270,400 / 8,309,664
Processed 5,770,400 / 8,309,664
Processed 6,148,800 / 8,309,664
Processed 6,648,800 / 8,309,664
Processed 7,027,200 / 8,309,664
Processed 7,527,200 / 8,309,664
Processed 7,905,600 / 8,309,664
Processed 8,309,664 / 8,309,664


In [21]:
calibration_table = []

for i in range(len(calibration)):
    rows = calibration[i]["rows"]

    if rows == 0:
        continue

    mean_predicted = (
        calibration[i]["predicted_probability_sum"] / rows
    )

    actual_rate = (
        calibration[i]["actual_positive"] / rows
    )

    calibration_table.append({
        "Probability Range": f"{bin_edges[i]:.0%}–{bin_edges[i+1]:.0%}",
        "Rows": rows,
        "Mean Predicted Probability": mean_predicted,
        "Actual Crime Rate": actual_rate
    })

calibration_df = pd.DataFrame(calibration_table)

calibration_df

,Probability Range,Rows,Mean Predicted Probability,Actual Crime Rate
0,0%–10%,12853900,0.027091,0.022992
1,10%–20%,2312070,0.142004,0.121144
2,20%–30%,905673,0.242944,0.218142
3,30%–40%,369739,0.343387,0.324651
4,40%–50%,143116,0.439631,0.426507
5,50%–60%,26752,0.533511,0.534577
6,60%–70%,4494,0.655469,0.711393
7,70%–80%,3564,0.739397,0.789562
8,80%–90%,20,0.804086,0.850000


In [22]:
mae_sum = 0.0
squared_error_sum = 0.0
poisson_deviance_sum = 0.0
zero_mae_sum = 0.0
positive_mae_sum = 0.0

total_rows = 0
zero_rows = 0
positive_rows = 0
actual_sum = 0.0
predicted_sum = 0.0

for batch in dataset.to_batches(batch_size=BATCH_SIZE):
    batch_df = batch.to_pandas()

    X_batch = batch_df.drop(
        columns=[TARGET, DATETIME_COLUMN]
    )

    y_batch = batch_df[TARGET].to_numpy()
    predicted_count = model.predict(X_batch)

    absolute_error = np.abs(y_batch - predicted_count)
    squared_error = (y_batch - predicted_count) ** 2

    mae_sum += absolute_error.sum()
    squared_error_sum += squared_error.sum()

    zero_mask = y_batch == 0
    positive_mask = y_batch >= 1
    actual_sum += y_batch.sum()
    predicted_sum += predicted_count.sum()
    zero_mae_sum += absolute_error[zero_mask].sum()
    positive_mae_sum += absolute_error[positive_mask].sum()

    zero_rows += int(zero_mask.sum())
    positive_rows += int(positive_mask.sum())
    total_rows += len(y_batch)

    safe_prediction = np.maximum(predicted_count, 1e-12)

    deviance = np.where(
        y_batch == 0,
        2 * safe_prediction,
        2 * (
            y_batch * np.log(y_batch / safe_prediction)
            - (y_batch - safe_prediction)
        )
    )

    poisson_deviance_sum += deviance.sum()

print("MAE:", mae_sum / total_rows)
print("RMSE:", np.sqrt(squared_error_sum / total_rows))
print("Zero MAE:", zero_mae_sum / zero_rows)
print("Positive MAE:", positive_mae_sum / positive_rows)
print("Poisson Deviance:", poisson_deviance_sum / total_rows)
print("Mean Actual:", actual_sum / total_rows)
print("Mean Predicted:", predicted_sum / total_rows)

C:\Users\sri16\AppData\Local\Temp\ipykernel_36568\1762043964.py:46: RuntimeWarning: divide by zero encountered in log
  y_batch * np.log(y_batch / safe_prediction)
C:\Users\sri16\AppData\Local\Temp\ipykernel_36568\1762043964.py:46: RuntimeWarning: invalid value encountered in multiply
  y_batch * np.log(y_batch / safe_prediction)
C:\Users\sri16\AppData\Local\Temp\ipykernel_36568\1762043964.py:46: RuntimeWarning: divide by zero encountered in log
  y_batch * np.log(y_batch / safe_prediction)
C:\Users\sri16\AppData\Local\Temp\ipykernel_36568\1762043964.py:46: RuntimeWarning: invalid value encountered in multiply
  y_batch * np.log(y_batch / safe_prediction)
C:\Users\sri16\AppData\Local\Temp\ipykernel_36568\1762043964.py:46: RuntimeWarning: divide by zero encountered in log
  y_batch * np.log(y_batch / safe_prediction)
C:\Users\sri16\AppData\Local\Temp\ipykernel_36568\1762043964.py:46: RuntimeWarning: invalid value encountered in multiply
  y_batch * np.log(y_batch / safe_prediction)
C:\U

MAE: 0.11591886349787352
RMSE: 0.27336431543759165
Zero MAE: 0.0653223865662119
Positive MAE: 0.9281114990606124
Poisson Deviance: 0.28208950237863567
Mean Actual: 0.06802838237502744
Mean Predicted: 0.07517829


C:\Users\sri16\AppData\Local\Temp\ipykernel_36568\1762043964.py:46: RuntimeWarning: divide by zero encountered in log
  y_batch * np.log(y_batch / safe_prediction)
C:\Users\sri16\AppData\Local\Temp\ipykernel_36568\1762043964.py:46: RuntimeWarning: invalid value encountered in multiply
  y_batch * np.log(y_batch / safe_prediction)


In [23]:
raw_probabilities = []
actual_outcomes = []

for batch in dataset.to_batches(batch_size=BATCH_SIZE):
    batch_df = batch.to_pandas()

    X_batch = batch_df.drop(
        columns=[TARGET, DATETIME_COLUMN]
    )

    predicted_count = model.predict(X_batch)

    predicted_probability = (
        1 - np.exp(-predicted_count)
    )

    raw_probabilities.append(predicted_probability)
    actual_outcomes.append(
        (batch_df[TARGET].to_numpy() >= 1).astype(np.int8)
    )

raw_probabilities = np.concatenate(raw_probabilities)
actual_outcomes = np.concatenate(actual_outcomes)

print("Rows:", len(raw_probabilities))
print("Probability range:", raw_probabilities.min(), "to", raw_probabilities.max())
print("Actual positive rate:", actual_outcomes.mean())

Rows: 8309664
Probability range: 1.7881393e-06 to 0.8071616
Actual positive rate: 0.05864292467180382


In [24]:
from sklearn.isotonic import IsotonicRegression

calibrator = IsotonicRegression(
    y_min=0.0,
    y_max=1.0,
    out_of_bounds="clip"
)

calibrator.fit(
    raw_probabilities,
    actual_outcomes
)

calibrated_probabilities = calibrator.predict(
    raw_probabilities
)

print("Raw mean probability:", raw_probabilities.mean())
print("Calibrated mean probability:", calibrated_probabilities.mean())
print("Actual positive rate:", actual_outcomes.mean())

Raw mean probability: 0.06721563
Calibrated mean probability: 0.05864342
Actual positive rate: 0.05864292467180382


In [25]:
bin_edges = np.arange(0.0, 1.1, 0.1)

calibration_rows = []

for i in range(len(bin_edges) - 1):
    mask = (
        (calibrated_probabilities >= bin_edges[i]) &
        (calibrated_probabilities < bin_edges[i + 1])
    )

    if not np.any(mask):
        continue

    calibration_rows.append({
        "Probability Range": f"{bin_edges[i]:.0%}–{bin_edges[i+1]:.0%}",
        "Rows": int(mask.sum()),
        "Mean Predicted Probability": calibrated_probabilities[mask].mean(),
        "Actual Crime Rate": actual_outcomes[mask].mean()
    })

calibrated_calibration_df = pd.DataFrame(calibration_rows)

calibrated_calibration_df

,Probability Range,Rows,Mean Predicted Probability,Actual Crime Rate
0,0%–10%,6776366,0.026213,0.026212
1,10%–20%,954946,0.142861,0.142860
2,20%–30%,342547,0.240248,0.240247
3,30%–40%,164919,0.345121,0.345121
4,40%–50%,54020,0.444150,0.444150
5,50%–60%,12568,0.543841,0.543842
6,60%–70%,1274,0.658556,0.658556
7,70%–80%,1861,0.759807,0.759807
8,80%–90%,1163,0.825451,0.825451


In [26]:
TEST_YEAR = 2025
test_path = yearly_path(TEST_YEAR)

test_dataset = ds.dataset(test_path, format="parquet")

bin_edges = np.arange(0.0, 1.1, 0.1)

raw_calibration = [
    {"rows": 0, "predicted_sum": 0.0, "actual_positive": 0}
    for _ in range(len(bin_edges) - 1)
]

calibrated_calibration = [
    {"rows": 0, "predicted_sum": 0.0, "actual_positive": 0}
    for _ in range(len(bin_edges) - 1)
]

processed_rows = 0

for batch in test_dataset.to_batches(batch_size=BATCH_SIZE):
    batch_df = batch.to_pandas()

    X_batch = batch_df.drop(
        columns=[TARGET, DATETIME_COLUMN]
    )

    y_batch = batch_df[TARGET].to_numpy()
    actual_positive = (y_batch >= 1)

    predicted_count = model.predict(X_batch)

    raw_probability = (
        1 - np.exp(-predicted_count)
    )

    calibrated_probability = calibrator.predict(
        raw_probability
    )

    for probabilities, calibration in [
        (raw_probability, raw_calibration),
        (calibrated_probability, calibrated_calibration)
    ]:
        bin_ids = np.digitize(
            probabilities,
            bin_edges,
            right=False
        ) - 1

        bin_ids = np.clip(
            bin_ids,
            0,
            len(calibration) - 1
        )

        for bin_id in range(len(calibration)):
            mask = bin_ids == bin_id

            if not np.any(mask):
                continue

            calibration[bin_id]["rows"] += int(mask.sum())

            calibration[bin_id]["predicted_sum"] += (
                probabilities[mask].sum()
            )

            calibration[bin_id]["actual_positive"] += int(
                actual_positive[mask].sum()
            )

    processed_rows += len(batch_df)

    print(f"Processed {processed_rows:,} / 8,286,960")

Processed 438,000 / 8,286,960
Processed 876,000 / 8,286,960
Processed 1,314,000 / 8,286,960
Processed 1,752,000 / 8,286,960
Processed 2,190,000 / 8,286,960
Processed 2,628,000 / 8,286,960
Processed 3,066,000 / 8,286,960
Processed 3,504,000 / 8,286,960
Processed 3,942,000 / 8,286,960
Processed 4,380,000 / 8,286,960
Processed 4,818,000 / 8,286,960
Processed 5,256,000 / 8,286,960
Processed 5,694,000 / 8,286,960
Processed 6,132,000 / 8,286,960
Processed 6,570,000 / 8,286,960
Processed 7,008,000 / 8,286,960
Processed 7,446,000 / 8,286,960
Processed 7,884,000 / 8,286,960
Processed 8,286,960 / 8,286,960


In [27]:
def make_calibration_table(calibration):
    rows = []

    for i, item in enumerate(calibration):
        if item["rows"] == 0:
            continue

        rows.append({
            "Probability Range": (
                f"{bin_edges[i]:.0%}–{bin_edges[i+1]:.0%}"
            ),
            "Rows": item["rows"],
            "Mean Predicted Probability": (
                item["predicted_sum"] / item["rows"]
            ),
            "Actual Crime Rate": (
                item["actual_positive"] / item["rows"]
            )
        })

    return pd.DataFrame(rows)


raw_2025_df = make_calibration_table(raw_calibration)
calibrated_2025_df = make_calibration_table(calibrated_calibration)

print("RAW 2025")
display(raw_2025_df)

print("CALIBRATED 2025")
display(calibrated_2025_df)

RAW 2025


,Probability Range,Rows,Mean Predicted Probability,Actual Crime Rate
0,0%–10%,6194952,0.029201,0.020864
1,10%–20%,1281714,0.141679,0.109478
2,20%–30%,477676,0.242088,0.202248
3,30%–40%,223043,0.344002,0.299045
4,40%–50%,88695,0.439754,0.410170
5,50%–60%,16347,0.533401,0.516731
6,60%–70%,1635,0.655294,0.626911
7,70%–80%,2877,0.743137,0.754258
8,80%–90%,21,0.804049,0.857143


CALIBRATED 2025


,Probability Range,Rows,Mean Predicted Probability,Actual Crime Rate
0,0%–10%,6571017,0.027703,0.024296
1,10%–20%,1062039,0.142940,0.129527
2,20%–30%,381670,0.241017,0.222409
3,30%–40%,194968,0.343364,0.322961
4,40%–50%,59760,0.444311,0.433618
5,50%–60%,13212,0.542809,0.519225
6,60%–70%,1264,0.658172,0.630538
7,70%–80%,1828,0.758986,0.730306
8,80%–90%,1202,0.825458,0.789517


In [28]:
tweedie_model = xgb.XGBRegressor(
    objective="reg:tweedie",
    tweedie_variance_power=1.5,
    n_estimators=600,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=20,
    subsample=0.9,
    colsample_bytree=0.9,
    tree_method="hist",
    enable_categorical=True,
    random_state=42,
    n_jobs=-1
)

tweedie_model.fit(
    X_train,
    y_train
)

print("Full-data Tweedie model trained successfully.")

Full-data Tweedie model trained successfully.


In [29]:
poisson_mae = 0.0
poisson_squared_error = 0.0
poisson_positive_mae = 0.0

tweedie_mae = 0.0
tweedie_squared_error = 0.0
tweedie_positive_mae = 0.0

total_rows = 0
positive_rows = 0

for batch in dataset.to_batches(batch_size=BATCH_SIZE):
    batch_df = batch.to_pandas()

    X_batch = batch_df.drop(
        columns=[TARGET, DATETIME_COLUMN]
    )

    y_batch = batch_df[TARGET].to_numpy()
    positive_mask = y_batch >= 1

    poisson_prediction = model.predict(X_batch)
    tweedie_prediction = tweedie_model.predict(X_batch)

    poisson_error = y_batch - poisson_prediction
    tweedie_error = y_batch - tweedie_prediction

    poisson_mae += np.abs(poisson_error).sum()
    poisson_squared_error += (poisson_error ** 2).sum()
    poisson_positive_mae += np.abs(
        poisson_error[positive_mask]
    ).sum()

    tweedie_mae += np.abs(tweedie_error).sum()
    tweedie_squared_error += (tweedie_error ** 2).sum()
    tweedie_positive_mae += np.abs(
        tweedie_error[positive_mask]
    ).sum()

    total_rows += len(y_batch)
    positive_rows += int(positive_mask.sum())

print("Poisson")
print("MAE:", poisson_mae / total_rows)
print("RMSE:", np.sqrt(poisson_squared_error / total_rows))
print("Positive MAE:", poisson_positive_mae / positive_rows)

print()

print("Tweedie")
print("MAE:", tweedie_mae / total_rows)
print("RMSE:", np.sqrt(tweedie_squared_error / total_rows))
print("Positive MAE:", tweedie_positive_mae / positive_rows)

Poisson
MAE: 0.11591886349787352
RMSE: 0.27336431543759165
Positive MAE: 0.9281114990606124

Tweedie
MAE: 0.11456495629310769
RMSE: 0.27345075823623805
Positive MAE: 0.9317094432723075


In [30]:
count_calibration = {
    "Poisson": {},
    "Tweedie": {}
}

for batch in dataset.to_batches(batch_size=BATCH_SIZE):
    batch_df = batch.to_pandas()

    X_batch = batch_df.drop(
        columns=[TARGET, DATETIME_COLUMN]
    )

    y_batch = batch_df[TARGET].to_numpy()

    poisson_prediction = model.predict(X_batch)
    tweedie_prediction = tweedie_model.predict(X_batch)

    for name, predictions in [
        ("Poisson", poisson_prediction),
        ("Tweedie", tweedie_prediction)
    ]:
        for actual_count in [0, 1, 2, 3]:
            mask = y_batch == actual_count

            if actual_count not in count_calibration[name]:
                count_calibration[name][actual_count] = {
                    "rows": 0,
                    "prediction_sum": 0.0
                }

            count_calibration[name][actual_count]["rows"] += int(mask.sum())
            count_calibration[name][actual_count]["prediction_sum"] += (
                predictions[mask].sum()
            )

        mask = y_batch >= 4

        if 4 not in count_calibration[name]:
            count_calibration[name][4] = {
                "rows": 0,
                "prediction_sum": 0.0
            }

        count_calibration[name][4]["rows"] += int(mask.sum())
        count_calibration[name][4]["prediction_sum"] += (
            predictions[mask].sum()
        )

In [31]:
count_calibration_rows = []

for name in ["Poisson", "Tweedie"]:
    for actual_count, values in count_calibration[name].items():
        mean_prediction = (
            values["prediction_sum"] / values["rows"]
        )

        count_calibration_rows.append({
            "Model": name,
            "Actual Crime Count": (
                "4+" if actual_count == 4 else actual_count
            ),
            "Rows": values["rows"],
            "Mean Predicted Count": mean_prediction
        })

count_calibration_df = pd.DataFrame(
    count_calibration_rows
)

count_calibration_df

,Model,Actual Crime Count,Rows,Mean Predicted Count
0,Poisson,0,7822361,0.065322
1,Poisson,1,422677,0.213821
2,Poisson,2,54134,0.336808
3,Poisson,3,8443,0.458071
4,Poisson,4+,2049,0.611856
5,Tweedie,0,7822361,0.063660
6,Tweedie,1,422677,0.210394
7,Tweedie,2,54134,0.330555
8,Tweedie,3,8443,0.446108
9,Tweedie,4+,2049,0.589809


In [32]:
bin_edges = np.arange(0.0, 1.1, 0.1)

poisson_calibration = [
    {
        "rows": 0,
        "predicted_sum": 0.0,
        "actual_positive": 0
    }
    for _ in range(len(bin_edges) - 1)
]

tweedie_calibration = [
    {
        "rows": 0,
        "predicted_sum": 0.0,
        "actual_positive": 0
    }
    for _ in range(len(bin_edges) - 1)
]

for batch in dataset.to_batches(batch_size=BATCH_SIZE):
    batch_df = batch.to_pandas()

    X_batch = batch_df.drop(
        columns=[TARGET, DATETIME_COLUMN]
    )

    y_batch = batch_df[TARGET].to_numpy()
    actual_positive = y_batch >= 1

    poisson_count = model.predict(X_batch)
    tweedie_count = tweedie_model.predict(X_batch)

    poisson_probability = (
        1 - np.exp(-poisson_count)
    )

    tweedie_probability = (
        1 - np.exp(-tweedie_count)
    )

    for probabilities, calibration in [
        (poisson_probability, poisson_calibration),
        (tweedie_probability, tweedie_calibration)
    ]:
        bin_ids = np.digitize(
            probabilities,
            bin_edges,
            right=False
        ) - 1

        bin_ids = np.clip(
            bin_ids,
            0,
            len(calibration) - 1
        )

        for bin_id in range(len(calibration)):
            mask = bin_ids == bin_id

            if not np.any(mask):
                continue

            calibration[bin_id]["rows"] += int(mask.sum())

            calibration[bin_id]["predicted_sum"] += (
                probabilities[mask].sum()
            )

            calibration[bin_id]["actual_positive"] += int(
                actual_positive[mask].sum()
            )

In [33]:
def build_probability_table(calibration):
    rows = []

    for i, item in enumerate(calibration):
        if item["rows"] == 0:
            continue

        rows.append({
            "Probability Range": (
                f"{bin_edges[i]:.0%}–{bin_edges[i+1]:.0%}"
            ),
            "Rows": item["rows"],
            "Mean Predicted Probability": (
                item["predicted_sum"] / item["rows"]
            ),
            "Actual Crime Rate": (
                item["actual_positive"] / item["rows"]
            )
        })

    return pd.DataFrame(rows)


poisson_probability_df = build_probability_table(
    poisson_calibration
)

tweedie_probability_df = build_probability_table(
    tweedie_calibration
)

print("POISSON")
display(poisson_probability_df)

print("TWEEDIE")
display(tweedie_probability_df)

POISSON


,Probability Range,Rows,Mean Predicted Probability,Actual Crime Rate
0,0%–10%,6433426,0.027849,0.022774
1,10%–20%,1162772,0.141780,0.121902
2,20%–30%,429733,0.242933,0.219681
3,30%–40%,183296,0.345209,0.322593
4,40%–50%,80696,0.439701,0.422078
5,50%–60%,15158,0.534875,0.533382
6,60%–70%,1694,0.654555,0.646399
7,70%–80%,2869,0.743156,0.785988
8,80%–90%,20,0.804086,0.850000


TWEEDIE


,Probability Range,Rows,Mean Predicted Probability,Actual Crime Rate
0,0%–10%,6499618,0.027356,0.023440
1,10%–20%,1106672,0.141431,0.124797
2,20%–30%,414325,0.242481,0.219871
3,30%–40%,196110,0.345291,0.322865
4,40%–50%,76250,0.438617,0.426308
5,50%–60%,12341,0.532394,0.541933
6,60%–70%,1985,0.657696,0.678086
7,70%–80%,2363,0.728646,0.800254


In [35]:
poisson_probabilities = []
actual_outcomes = []

processed_rows = 0

for batch in dataset.to_batches(batch_size=BATCH_SIZE):
    batch_df = batch.to_pandas()

    X_batch = batch_df.drop(
        columns=[TARGET, DATETIME_COLUMN]
    )

    y_batch = batch_df[TARGET].to_numpy()

    predicted_count = model.predict(X_batch)

    predicted_probability = (
        1 - np.exp(-predicted_count)
    )

    poisson_probabilities.append(predicted_probability)
    actual_outcomes.append(
        (y_batch >= 1).astype(np.int8)
    )

    processed_rows += len(batch_df)

    print(f"Processed {processed_rows:,} / 8,309,664")

Processed 500,000 / 8,309,664
Processed 878,400 / 8,309,664
Processed 1,378,400 / 8,309,664
Processed 1,756,800 / 8,309,664
Processed 2,256,800 / 8,309,664
Processed 2,635,200 / 8,309,664
Processed 3,135,200 / 8,309,664
Processed 3,513,600 / 8,309,664
Processed 4,013,600 / 8,309,664
Processed 4,392,000 / 8,309,664
Processed 4,892,000 / 8,309,664
Processed 5,270,400 / 8,309,664
Processed 5,770,400 / 8,309,664
Processed 6,148,800 / 8,309,664
Processed 6,648,800 / 8,309,664
Processed 7,027,200 / 8,309,664
Processed 7,527,200 / 8,309,664
Processed 7,905,600 / 8,309,664
Processed 8,309,664 / 8,309,664


In [36]:
poisson_probabilities = np.concatenate(
    poisson_probabilities
)

actual_outcomes = np.concatenate(
    actual_outcomes
)

print("Rows:", len(poisson_probabilities))
print("Mean probability:", poisson_probabilities.mean())
print("Actual positive rate:", actual_outcomes.mean())

Rows: 8309664
Mean probability: 0.06721563
Actual positive rate: 0.05864292467180382


In [37]:
for top_fraction in [0.01, 0.05, 0.10]:
    threshold = np.quantile(
        poisson_probabilities,
        1 - top_fraction
    )

    top_mask = poisson_probabilities >= threshold

    print(
        f"Top {top_fraction:.0%}:",
        f"{top_mask.sum():,} rows",
        f"Actual crime rate: {actual_outcomes[top_mask].mean():.4%}"
    )

print(
    f"Overall crime rate: {actual_outcomes.mean():.4%}"
)

Top 1%: 83,099 rows Actual crime rate: 46.8080%
Top 5%: 415,486 rows Actual crime rate: 33.1932%
Top 10%: 830,967 rows Actual crime rate: 26.3757%
Overall crime rate: 5.8643%


In [38]:
from sklearn.metrics import roc_auc_score, average_precision_score

roc_auc = roc_auc_score(
    actual_outcomes,
    poisson_probabilities
)

pr_auc = average_precision_score(
    actual_outcomes,
    poisson_probabilities
)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")

ROC-AUC: 0.8412
PR-AUC: 0.2694


In [39]:
grid_ids = []

for batch in dataset.to_batches(batch_size=BATCH_SIZE):
    batch_df = batch.to_pandas()
    grid_ids.append(batch_df["grid_id"].to_numpy())

grid_ids = np.concatenate(grid_ids)

print("Rows:", len(grid_ids))
print("Unique grids:", len(np.unique(grid_ids)))

Rows: 8309664
Unique grids: 946


In [40]:
grid_results = pd.DataFrame({
    "grid_id": grid_ids,
    "predicted_probability": poisson_probabilities,
    "actual_outcome": actual_outcomes
})

grid_summary = (
    grid_results
    .groupby("grid_id", observed=True)
    .agg(
        observations=("actual_outcome", "size"),
        mean_predicted_probability=("predicted_probability", "mean"),
        actual_crime_rate=("actual_outcome", "mean")
    )
    .sort_values("mean_predicted_probability", ascending=False)
)

print(grid_summary.head(20))

              observations  mean_predicted_probability  actual_crime_rate
grid_id                                                                  
40.75_-73.99          8784                    0.573939           0.552596
40.75_-74.0           8784                    0.410140           0.393898
40.86_-73.9           8784                    0.382770           0.354281
40.79_-73.95          8784                    0.376663           0.327413
40.74_-73.99          8784                    0.364947           0.304645
40.69_-73.99          8784                    0.358246           0.332309
40.85_-73.91          8784                    0.356450           0.329918
40.81_-73.92          8784                    0.355173           0.349841
40.83_-73.92          8784                    0.353478           0.334699
40.72_-74.0           8784                    0.349845           0.346311
40.76_-73.99          8784                    0.346646           0.393101
40.8_-73.95           8784            

In [41]:
from scipy.stats import spearmanr

correlation, p_value = spearmanr(
    grid_summary["mean_predicted_probability"],
    grid_summary["actual_crime_rate"]
)

print(f"Spearman correlation: {correlation:.4f}")
print(f"P-value: {p_value:.6e}")

Spearman correlation: 0.9957
P-value: 0.000000e+00


In [45]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data/yearly")

val_2024 = pd.read_parquet(
    DATA_DIR / "citysense_2024.parquet"
)

X_val = val_2024.drop(columns=["crime_count", "cmplnt_fr_dt"])
y_val_binary = (val_2024["crime_count"] >= 1).astype(int)

print("X_val:", X_val.shape)
print("y_val:", y_val_binary.shape)

X_val: (8309664, 14)
y_val: (8309664,)


In [46]:
binary_model = xgb.XGBClassifier(
    objective="binary:logistic",
    n_estimators=600,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=20,
    subsample=0.9,
    colsample_bytree=0.9,
    tree_method="hist",
    enable_categorical=True,
    random_state=42,
    n_jobs=-1
)

binary_model.fit(X_train, y_train_binary)

,"objective objective: typing.Union[str, xgboost.objective.Objective, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.9
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets imp

In [47]:
binary_probabilities = binary_model.predict_proba(X_val)[:, 1]

print("Mean probability:", binary_probabilities.mean())
print("Actual crime rate:", y_val_binary.mean())
print(
    "Probability range:",
    binary_probabilities.min(),
    "to",
    binary_probabilities.max()
) 

Mean probability: 0.06389137
Actual crime rate: 0.05864292467180382
Probability range: 9.0346856e-07 to 0.7746695


In [48]:
from sklearn.metrics import roc_auc_score, average_precision_score

roc_auc = roc_auc_score(
    y_val_binary,
    binary_probabilities
)

pr_auc = average_precision_score(
    y_val_binary,
    binary_probabilities
)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")

ROC-AUC: 0.8409
PR-AUC: 0.2695


In [49]:
bins = np.arange(0, 1.01, 0.10)

calibration = pd.DataFrame({
    "probability": binary_probabilities,
    "actual": y_val_binary.to_numpy()
})

calibration["bin"] = pd.cut(
    calibration["probability"],
    bins=bins,
    include_lowest=True
)

calibration_summary = (
    calibration
    .groupby("bin", observed=True)
    .agg(
        rows=("actual", "size"),
        mean_predicted=("probability", "mean"),
        actual_rate=("actual", "mean")
    )
)

print(calibration_summary)

                  rows  mean_predicted  actual_rate
bin                                                
(-0.001, 0.1]  6555189        0.028025     0.024023
(0.1, 0.2]     1104201        0.141085     0.129234
(0.2, 0.3]      408288        0.242745     0.231285
(0.3, 0.4]      168422        0.343604     0.341292
(0.4, 0.5]       57890        0.439751     0.443358
(0.5, 0.6]       11399        0.526096     0.557768
(0.6, 0.7]        1790        0.655718     0.680447
(0.7, 0.8]        2485        0.729486     0.794769


In [50]:
val_2025 = pd.read_parquet(
    DATA_DIR / "citysense_2025.parquet"
)

X_2025 = val_2025.drop(
    columns=["crime_count", "cmplnt_fr_dt"]
)

y_2025 = (val_2025["crime_count"] >= 1).astype(int)

binary_probabilities_2025 = binary_model.predict_proba(X_2025)[:, 1]

print("Rows:", len(y_2025))
print("Mean probability:", binary_probabilities_2025.mean())
print("Actual crime rate:", y_2025.mean())
print(
    "Probability range:",
    binary_probabilities_2025.min(),
    "to",
    binary_probabilities_2025.max()
)

Rows: 8286960
Mean probability: 0.06805733
Actual crime rate: 0.058033464623939296
Probability range: 8.456794e-07 to 0.7746695


In [51]:
from sklearn.metrics import roc_auc_score, average_precision_score

roc_auc_2025 = roc_auc_score(
    y_2025,
    binary_probabilities_2025
)

pr_auc_2025 = average_precision_score(
    y_2025,
    binary_probabilities_2025
)

print(f"ROC-AUC: {roc_auc_2025:.4f}")
print(f"PR-AUC: {pr_auc_2025:.4f}")

ROC-AUC: 0.8384
PR-AUC: 0.2626


In [52]:
calibration_2025 = pd.DataFrame({
    "probability": binary_probabilities_2025,
    "actual": y_2025.to_numpy()
})

calibration_2025["bin"] = pd.cut(
    calibration_2025["probability"],
    bins=np.arange(0, 1.01, 0.10),
    include_lowest=True
)

summary_2025 = (
    calibration_2025
    .groupby("bin", observed=True)
    .agg(
        rows=("actual", "size"),
        mean_predicted=("probability", "mean"),
        actual_rate=("actual", "mean")
    )
)

print(summary_2025)

                  rows  mean_predicted  actual_rate
bin                                                
(-0.001, 0.1]  6461333        0.030536     0.023510
(0.1, 0.2]     1116324        0.141840     0.121150
(0.2, 0.3]      441937        0.242817     0.217943
(0.3, 0.4]      191203        0.342146     0.323944
(0.4, 0.5]       60212        0.439795     0.435345
(0.5, 0.6]       11688        0.526100     0.533710
(0.6, 0.7]        1766        0.657346     0.657984
(0.7, 0.8]        2497        0.729893     0.762915
